## 2 Pizzas

- Harina 500 gr
- condimento pizza 3 gr
- sal 10 gr
- pimenton 2 gr
- Oregano 5 gr
- 1 lata tomate
- 1 sobre levadura
- Aji molido 2 gr
- 1/6 ajo
- Queso mozzarela 600 gr


In [0]:
WITH cantidades_receta_pizza AS (
  SELECT * FROM VALUES
    ('Harina Trigo 0000 MORIXE 1kg', 500),
    ('Condimento Para Pizza La Campagnola Sob 23 Grm', 3),
    ('Sal Fina CELUSAL 500g', 10),
    ('Pimenton Seleccionado Dos Anclas Pou 50 Grm', 2),
    ('Orégano Dos Anclas 25 Grm', 5),
    ('Tomate Perita ARCOR Lata 400 Gr', 400),
    ('Levadura Seca CALSA Mi Pan 10 Gr', 10),
    ('Aji Molido DOS ANCLAS Sobre 50 Gr',2),
    ('Ajo', 1.0/6.0),
    ('Queso Mozzarella BARRAZA 1kg', 600)
  AS t(nombre, cantidad_receta)
)

SELECT * FROM cantidades_receta_pizza

In [0]:
WITH receta_pizza(
  SELECT 
  nombre,
  precio_normalizado AS precio,
  CAST(CASE 
    WHEN nombre in ("Harina Trigo 0000 MORIXE 1kg","Queso Mozzarella BARRAZA 1kg") then '1000'
    ELSE NULLIF(REGEXP_EXTRACT(trim(nombre), '(\\d+)', 0),'') 
  END AS INT) AS peso_en_gramos,

  CAST(CASE 
    WHEN nombre = "Ajo" THEN '1'
    ELSE NULL
  END AS INT) AS unidades,
  fecha_extraccion,
  url
  FROM supermercadoetl_prod.silver.precios_productos
  WHERE nombre IN ("Harina Trigo 0000 MORIXE 1kg",
                  "Condimento Para Pizza La Campagnola Sob 23 Grm",
                  "Sal Fina CELUSAL 500g",
                  "Pimenton Seleccionado Dos Anclas Pou 50 Grm",
                  "Orégano Dos Anclas 25 Grm",
                  "Tomate Perita ARCOR Lata 400 Gr",
                  "Levadura Seca CALSA Mi Pan 10 Gr",
                  "Aji Molido DOS ANCLAS Sobre 50 Gr",
                  "Ajo",
                  "Queso Mozzarella BARRAZA 1kg")
),
precios_unitarios AS (
  SELECT 
  nombre,
  precio,
  peso_en_gramos,
  ROUND(precio / peso_en_gramos,2) AS precio_por_gramo,
  CASE 
    WHEN nombre = "Ajo" THEN ROUND(precio * unidades)
    ELSE NULL
  END AS precio_por_unidad,
  unidades,
  fecha_extraccion,
  url
  FROM receta_pizza
),
cantidades_receta_pizza AS (
  SELECT * FROM VALUES
    ('Harina Trigo 0000 MORIXE 1kg', 500),
    ('Condimento Para Pizza La Campagnola Sob 23 Grm', 3),
    ('Sal Fina CELUSAL 500g', 10),
    ('Pimenton Seleccionado Dos Anclas Pou 50 Grm', 2),
    ('Orégano Dos Anclas 25 Grm', 5),
    ('Tomate Perita ARCOR Lata 400 Gr', 400),
    ('Levadura Seca CALSA Mi Pan 10 Gr', 10),
    ('Aji Molido DOS ANCLAS Sobre 50 Gr',2),
    ('Ajo', 1.0/6.0),
    ('Queso Mozzarella BARRAZA 1kg', 600)
  AS t(nombre, cantidad_receta)
),
costo_pizza AS(

SELECT 
  p.nombre,
  p.precio,
  COALESCE(precio_por_gramo, precio_por_unidad) AS precio_unitario,
  c.cantidad_receta,
  ROUND(COALESCE(precio_por_gramo, precio_por_unidad) * c.cantidad_receta,2) AS costo,
  p.fecha_extraccion,
  p.url
FROM precios_unitarios p
LEFT JOIN cantidades_receta_pizza c
ON TRIM(LOWER(p.nombre))  = TRIM(LOWER(c.nombre)) 
)
SELECT 
fecha_extraccion,
SUM(costo) AS costo_pizza 
FROM costo_pizza
GROUP BY fecha_extraccion